In [14]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
from numpy.linalg import norm, pinv

In [8]:
x = np.loadtxt(r'..\Data\Kuramoto\Time_series.csv', delimiter=',')
x_dot = np.loadtxt(r'..\Data\Kuramoto\Time_series_derivative.csv', delimiter=',')

In [9]:
Gamma = 1
r = 2


$\begin{equation}
\frac{d}{dt}x_i=f_ix_i+\sum_{\substack{j=1 \\ j\neq{i}}}^N\epsilon_{ij}K(x_i,x_j)
\tag{1}
\end{equation}
$  
&emsp; Assuming $f_i$ and $K$ smooth enough, we can represent them on new orthogonal basis (like Fourier basis):  
$\begin{equation}
x_i(t) = B_i^TZ_i\Lambda + A_i^TF_i + \eta_i
\tag{2}
\end{equation}$

&emsp; where $Z_i\Lambda \in \mathbb{R}^{(N-1)}$ represents coupling function to other oscillators, $F_i$ represents external forcing, and $\eta_i$ is approximation error.
    
&emsp; So the problem is to find coupling strength $B_i$, self-dynamic coeff $A_i$ and coupling function's coefficient on basis $Z_i:\Lambda$. 

In [10]:
N = x.shape[1]
K_est = np.zeros((N, N))

# Initialize and normalize the Fourier Coefficients
Lambda_initial = np.ones(2 * r)
Lambda_initial = (1 / norm(Lambda_initial)) * Lambda_initial

Iteration_number = np.zeros((N, Gamma))
Lambda_hat = Lambda_initial
Lambda_each_node = np.zeros((N * Gamma, 2 * r))
def est_F(x):
    """ 
    this function is used to estimate the F matrix which is the basis of the Fourier(or whatever) Coefficients
    in this code we set it 1 because for the Kuramoto model, all oscillators are single oscillators
    """
    return np.ones((x.shape[0], 1))

def est_Z(x, r, node_index):
    """ 
    this function is used to estimate the Z matrix which is the coupling basis
    
    x: (M, N): the time series data
    r: the number of Fourier Coefficients
    node_index: the index of the node that we want to estimate the Z matrix
    
    return:  
    Z: (M * (N - 1), r ^ 2): coupling basis
    
    formula: $$Z_i = [sin(k * (x_i - x_j)), cos(k * (x_i - x_j))]_{j=1}^{N}$$ where i != j
    then we have Z = [Z_1, Z_2, ..., Z_N] so Z.shape = (M * (N - 1), r ^ 2)
    """
    M, N = x.shape
    Z = np.zeros((M * (N - 1), 2 * r))
    
    x_temp = np.delete(x, node_index, axis=1)
    assert x_temp.shape == (M, N - 1)
    for t in range(M):
        Z_i = np.zeros((N - 1, 2 * r))
        for k in range(r):
            Diff = x[t, node_index] - x_temp[t, :]
            assert Diff.shape == (N - 1,)
            Z_i[:, 2 * k:2 * k + 2] = np.column_stack((np.sin(k * Diff), np.cos(k * Diff)))
        
        Z[t * (N - 1):(t + 1) * (N - 1), :] = Z_i # update node_i's Z matrix

    return Z

def test_est_Z():
    M = 5
    N = 3
    r = 2
    node_index = 1
    
    x = np.random.rand(M, N)
    
    Z = est_Z(x, r, node_index)
    print("M:", M)
    print("N:", N)
    print("r:", r)
    print("Z:", Z.shape)
    assert Z.shape == (M * (N - 1), 2*r)

test_est_Z()

M: 5
N: 3
r: 2
Z: (10, 4)


In [13]:
def estimate_topology(x_dot, F, Z, Lambda, N):
    """ 
    this step is the 4th step in the pseudo code
    using the optimization problem to estimate the coupling matrix, compute all nodes and all time points
    x_dot(M, N): the derivative of the time series data, M is the number of time points
    F(r, N): the Fourier basis
    Z(N - 1, r ^ 2): the coupling basis
    Lambda(2 * r, 1): the Fourier Coefficients
    N: the number of nodes
    
    A_hat: self dynamic f_i's basis coefficients
    B_hat: strength of coupling
    """
    M = x_dot.shape[0]
    X = np.zeros((M, N-1))
    for i in range(M):
        X[i, :] = (Z[(i)*(N-1):(i+1)*(N-1), :] @ Lambda).T
    
    Temp = np.eye(M) - (F @ np.linalg.pinv(F))
    assert Temp.shape == (M, M)
    
    # Define the optimization variables, this problem is a non-negative least square problem
    B_hat = cp.Variable((X.shape[1], x_dot.shape[1]), nonneg=True)
    constraints = [B_hat >= 0]
    # Define the objective function: ||X @ B_hat - x_dot||_2
    objective = cp.Minimize(cp.norm(Temp @ X @ B_hat - Temp @ x_dot, 'fro'))
    assert (Temp @ X @ B_hat).shape == x_dot.shape
    assert (Temp @ x_dot).shape == x_dot.shape
    problem = cp.Problem(objective, constraints)
    
    # Solve the problem
    problem.solve()
    
    B_hat_value = B_hat.value
    A_hat = np.linalg.pinv(F) @ (x_dot - X @ B_hat_value)
    
    return B_hat_value, A_hat


def test_estimate_topology():
    M = 5
    N = 3
    r = 2
    Lambda = np.random.rand(2 * r, 1)
    F = est_F(x_dot)
    Z = est_Z(x_dot, r, 0)
    B_hat, A_hat = estimate_topology(x_dot, F, Z, Lambda, N)
    print("B_hat:", B_hat.shape)
    print("A_hat:", A_hat.shape)
    
test_estimate_topology()

B_hat: (2, 100)
A_hat: (1, 100)


In [12]:
def estimate_lambda(x_dot, F, Z, B_hat, A_hat, N, r):
    """ 
    this step is 5th step in the pseudo code
    to estimate the \Lambda matrix which is the coefficients of B^T * Z
    
    x_dot(M, N): the derivative of the time series data, M is the number of time points
    F(r, N): the Fourier basis
    Z(N - 1, r ^ 2): the coupling basis
    B_hat(N - 1, N): the strength(coefficient) of coupling(B^T * Z * \Lambda indicates coupling)
    A_hat(r, N): the self dynamic f_i's basis coefficients(A^T * F indicates self dynamic)
    F(r, N): the Fourier basis
    """
    M = x_dot.shape[0]
    X = np.zeros((M, 2 * r))
    
    for i in range(M):
        X[i, :] = B_hat.T @ Z[(i)*(N-1):(i+1)*(N-1), :]
    
    Lambda_hat = pinv(X) @ (x_dot - F @ A_hat)
    return Lambda_hat    

In [ ]:
class SINC_curamoto:
    def __init__(self, X, X_dot, r, epsilon, Gamma, max_iter) -> None:
        self.M = X.shape[0]
        self.N = X.shape[1]
        self.r = r
        self.epsilon = epsilon
        self.Gamma = Gamma
        self.X = X
        self.X_dot = X_dot
        assert self.X.shape == self.X_dot.shape
        self.K_est = np.zeros((self.N, self.N))
        self.Lambda_hat = np.ones(2 * r, 1)
        self.Lambda_hat /= np.linalg.norm(self.Lambda_hat)
        self.Lambda_each_node = np.zeros((self.N * self.Gamma, 2 * r))
        self.Iteration_number = np.zeros((self.N, self.Gamma))
        self.iter_index = 0
        self.max_iter = max_iter
        
    def step_optimization(self, F, Z, i):
        """ 
        this function is used to optimize the SINC algorithm
        """
        #step 4:
        B_hat, A_hat = estimate_topology(self.X_dot[:, i], F, Z, self.Lambda_hat, self.N)
        #step 5:
        Lambda_hat_update = estimate_lambda(self.X_dot[:, i], F, Z, B_hat, A_hat, self.N, self.r)
        #step 6 for avoiding Remark1:
        c = np.linalg.norm(Lambda_hat_update)
        assert c != 0
        Lambda_hat_update = Lambda_hat_update / c
        B_hat = c * B_hat
        #step 7:
        if (norm(Lambda_hat_update - self.Lambda_hat) < self.epsilon) or (self.Iteration_number[i] > 100):
            self.K_est[i, :] = B_hat
            self.Lambda_each_node[i * self.Gamma:(i + 1) * self.Gamma, :] = Lambda_hat_update
            self.Iteration_number[i] = self.Iteration_number[i] + 1
            self.Lambda_hat = Lambda_hat_update
        else:
            self.Lambda_hat = Lambda_hat_update


    def fit(self):
        
        for i in range(self.N):
            # prepare for step 4
            F = est_F(self.X_dot)
            Z = est_Z(self.X_dot, self.r, i)
            self.step_optimization(F, Z, i)
            self.iter_index += 1
            if self.iter_index > self.max_iter:
                break
        
        #step 16:
        self.Lambda_hat = np.mean(self.Lambda_each_node, axis=0)
        #step 17:
        for i in range(self.N):
            F = est_F(self.X_dot)
            Z = est_Z(self.X_dot, self.r, i)
            self.step_optimization(F, Z, i)
            self.iter_index += 1
            if self.iter_index > self.max_iter:
                break
            
            


        